# Chest X-ray Grad-CAM 시각화

DenseNet121 기반 U-multiclass 모델의 예측 결과를 Grad-CAM으로 시각화하여, 각 소견의 예측에 기여한 이미지 영역을 확인한다.

### 탐지 대상
- 심비대 (Cardiomegaly)
- 폐부종 (Edema)
- 흉수 (Pleural Effusion)

### 라벨 규칙
- 0: 음성 (Negative)
- 1: 양성 (Positive)
- 2: 불확실 (Uncertain)

### 시각화 구성
하나의 흉부 X-ray 이미지에 대해 총 4개의 이미지를 출력한다.

1. 원본 X-ray
2. 심비대 Grad-CAM Overlay
3. 폐부종 Grad-CAM Overlay
4. 흉수 Grad-CAM Overlay

현재는 무작위 초기화 모델과 임시 데이터를 이용하여 Grad-CAM 코드의 동작을 검증한다.

실제 모델 학습이 완료되면 최적 checkpoint와 X-ray 이미지를 연결하여 시각화를 진행한다.

## 2. 라이브러리 및 실행 환경

Grad-CAM 구현에 필요한 라이브러리를 불러오고 CPU/GPU 실행 환경을 설정한다.

- PyTorch: 모델 연산 및 기울기 계산
- torchvision: DenseNet121 모델 구성
- NumPy: 이미지 및 히트맵 배열 처리
- Matplotlib: Grad-CAM 결과 시각화

In [ ]:
import torch
import torch.nn.functional as F
from torch import nn
from torchvision.models import densenet121, DenseNet121_Weights

import numpy as np
import matplotlib.pyplot as plt

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

FINDING_NAMES = ["Cardiomegaly", "Edema", "Pleural Effusion"]
CLASS_NAMES = ["Negative", "Positive", "Uncertain"]

print("사용 장치:", device)

## 3. DenseNet121 모델 정의

기존 모델 학습 코드와 동일한 DenseNet121 기반 U-multiclass 모델 구조를 사용한다.

- Backbone: DenseNet121
- 출력 형태: (B, 3, 3)
- 소견 수: 3개
- 클래스 수: 소견별 3개

현재는 `pretrained=False`로 모델을 생성하여 코드 작동을 테스트한다.

실제 시각화에서는 학습이 완료된 모델의 checkpoint를 불러와 사용한다.

In [ ]:
class CXRModel(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = DenseNet121_Weights.DEFAULT if pretrained else None

        self.backbone = densenet121(weights=weights)
        self.backbone.classifier = nn.Linear(
            self.backbone.classifier.in_features, 9
        )

    def forward(self, images):
        return self.backbone(images).reshape(-1, 3, 3)

## 4. Grad-CAM 구현

Grad-CAM(Gradient-weighted Class Activation Mapping)을 이용하여 특정 소견과 클래스의 예측 점수에 기여한 영역을 시각화한다.

### 구현 과정

1. DenseNet121의 마지막 특징 추출 계층에서 Feature Map 추출
2. 선택한 소견 및 클래스의 출력 점수에 대한 Gradient 계산
3. Gradient의 공간 평균으로 채널별 가중치 계산
4. Feature Map과 가중치를 결합하여 CAM 생성
5. ReLU 적용 및 입력 이미지 크기로 확대
6. 히트맵을 0~1 범위로 정규화

### Grad-CAM 대상

- 소견 인덱스: 0=심비대, 1=폐부종, 2=흉수
- 기본 클래스 인덱스: 1=양성

Grad-CAM은 모델 예측에 기여한 영역을 보여주며, 실제 병변 위치를 정확히 나타내는 것은 아니다.

In [ ]:
class GradCAM:
    def __init__(self, model):
        self.model = model
        self.target_layer = model.backbone.features.norm5

        self.activations = None
        self.gradients = None

        self.forward_handle = self.target_layer.register_forward_hook(
            self.save_activations
        )

    def save_activations(self, module, inputs, output):
        # DenseNet의 이후 연산에서 원본 출력이 변경될 수 있으므로
        # clone한 텐서에 hook을 등록
        captured = output.clone()
        self.activations = captured.detach()
        captured.register_hook(self.save_gradients)
        return captured

    def save_gradients(self, grad):
        self.gradients = grad.detach()

    def generate(self, image, finding_idx, class_idx=1):
        self.model.eval()

        device = next(self.model.parameters()).device
        image = image.unsqueeze(0).to(device)

        self.model.zero_grad(set_to_none=True)
        self.activations = None
        self.gradients = None

        with torch.enable_grad():
            logits = self.model(image)

            score = logits[0, finding_idx, class_idx]
            score.backward()

        activations = self.activations
        gradients = self.gradients

        weights = gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * activations).sum(dim=1, keepdim=True)

        cam = F.relu(cam)
        cam = F.interpolate(
            cam,
            size=image.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        cam = cam[0, 0].detach().cpu().numpy()

        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)

        return cam

    def close(self):
        self.forward_handle.remove()

## 5. Grad-CAM 결과 시각화

하나의 X-ray 이미지에 대해 원본과 세 소견별 Grad-CAM Overlay를 나란히 출력한다.

- Original X-ray: 원본 흉부 X-ray
- Cardiomegaly: 심비대 양성 클래스 Grad-CAM Overlay
- Edema: 폐부종 양성 클래스 Grad-CAM Overlay
- Pleural Effusion: 흉수 양성 클래스 Grad-CAM Overlay

각 Overlay는 원본 이미지 위에 Grad-CAM 히트맵을 반투명하게 겹쳐 표시한다.

히트맵의 색상은 해당 클래스 점수에 대한 상대적 기여도를 나타낸다.

In [ ]:
def show_gradcam(image, cams):
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])

    # 이미지 정규화 해제
    img = image.detach().cpu().permute(1, 2, 0).numpy()
    img = np.clip(img * std + mean, 0, 1)

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))

    # 1. 원본 X-ray
    axes[0].imshow(img)
    axes[0].set_title("Original X-ray")

    # 2~4. 소견별 Grad-CAM Overlay
    for i, finding in enumerate(FINDING_NAMES):
        axes[i + 1].imshow(img)
        axes[i + 1].imshow(
            cams[i],
            cmap="jet",
            alpha=0.4,
            vmin=0,
            vmax=1
        )
        axes[i + 1].set_title(f"{finding} - Positive")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## 6. Grad-CAM 동작 테스트

무작위 초기화된 DenseNet121 모델과 임시 입력 데이터를 이용하여 Grad-CAM 구현이 정상적으로 작동하는지 확인한다.

### 확인 항목

- 세 소견에 대한 Grad-CAM 생성 여부
- 히트맵 출력 크기: (224, 224)
- 원본 이미지 및 소견별 Overlay 총 4개 출력 여부
- 소견별 Grad-CAM 생성 과정에서 오류 발생 여부

현재 사용하는 임시 이미지는 무작위 데이터이므로 실제 X-ray 영상이 아니다.

따라서 생성된 히트맵은 코드 동작 검증 목적으로만 사용하며, 의학적 의미를 해석하지 않는다.

In [ ]:
# 1. 임시 모델 생성
model = CXRModel(pretrained=False).to(device)

# 2. 임시 이미지 생성
image = torch.randn(3, 224, 224)

# 3. Grad-CAM 객체 생성
gradcam = GradCAM(model)

try:
    cams = []

    # 4. 세 소견별 Grad-CAM 생성
    for finding_idx in range(3):
        cam = gradcam.generate(
            image,
            finding_idx=finding_idx,
            class_idx=1
        )
        cams.append(cam)

    # 5. 원본 + 소견별 Overlay 출력
    show_gradcam(image, cams)

finally:
    gradcam.close()

## 실제 X-ray 이미지 기반 Grad-CAM 동작 검증

GPU 서버에 저장된 MIMIC-CXR-JPG 이미지를 이용하여
Grad-CAM 시각화 코드의 정상 동작 여부를 확인한다.

현재는 학습되지 않은 모델을 사용하므로
생성된 히트맵에 의학적 의미를 부여하지 않는다.

데이터 접근 제한을 준수하기 위해 실제 X-ray 이미지 및
해당 이미지가 포함된 노트북 실행 결과는 GitHub에 업로드하지 않는다.

In [ ]:
from pathlib import Path
from PIL import Image
from torchvision import transforms

image_dir = Path("../data/images")

print("현재 작업 경로:", Path.cwd())
print("이미지 폴더 존재:", image_dir.exists())

image_files = sorted(image_dir.rglob("*.jpg"))
print("이미지 수:", len(image_files))

if not image_files:
    raise FileNotFoundError("JPG 이미지가 없습니다.")

image_path = image_files[0]
print("선택한 이미지:", image_path)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

image = transform(Image.open(image_path).convert("RGB"))

print("이미지 형태:", image.shape)

In [ ]:
# 실제 X-ray 이미지 기반 Grad-CAM 테스트
gradcam = GradCAM(model)

try:
    cams = [
        gradcam.generate(image, finding_idx=i, class_idx=1)
        for i in range(3)
    ]

    show_gradcam(image, cams)

finally:
    gradcam.close()